# 01 – Interaction Extraction
Extract celebrity ↔ politician interactions from raw tweet JSON files.

**Outputs** saved to `outputs/interactions/`.

In [ ]:
import logging
import sys
sys.path.insert(0, '..')

import yaml
import pandas as pd

from src.data.loaders import (
    load_config, load_politicians, load_celebrity_engagement,
    build_politician_party_map, list_tweet_files,
)
from src.interactions.extractor import (
    extract_interactions, log_interaction_summary,
)

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
cfg = load_config('configs/config.yaml')
print('Config loaded.')

## 1. Load metadata

In [ ]:
df_pol = load_politicians(cfg)
df_engage = load_celebrity_engagement(cfg)

pol_map = build_politician_party_map(df_pol)   # {screen_name: party}
celeb_screens = set(df_engage['screen_name'])
pol_screens   = set(pol_map.keys())

print(f'Celebrities: {len(celeb_screens)}')
print(f'Politicians: {len(pol_screens)}')

## 2. Celebrity → Politician interactions

In [ ]:
all_files = list_tweet_files(cfg['data']['tweets_drive'])
date_start = cfg['date_filter']['start']
date_end   = cfg['date_filter']['end']

celeb_interactions = extract_interactions(
    source_files   = all_files,
    source_screens = celeb_screens,
    target_screens = pol_screens,
    party_map      = pol_map,
    date_start     = date_start,
    date_end       = date_end,
)
log_interaction_summary(celeb_interactions, label='Celeb→Pol')
celeb_interactions.head()

In [ ]:
import os
os.makedirs(cfg['outputs']['interactions_dir'], exist_ok=True)
out = cfg['outputs']['interactions_dir'] + 'celeb_interactions.json'
celeb_interactions.to_json(out)
print(f'Saved → {out}')

## 3. Politician → Celebrity interactions

In [ ]:
pol_interactions = extract_interactions(
    source_files   = all_files,
    source_screens = pol_screens,
    target_screens = celeb_screens,
    party_map      = pol_map,
    date_start     = date_start,
    date_end       = date_end,
)
log_interaction_summary(pol_interactions, label='Pol→Celeb')
pol_interactions.head()

In [ ]:
out = cfg['outputs']['interactions_dir'] + 'pol_interactions.json'
pol_interactions.to_json(out)
print(f'Saved → {out}')

## 4. Language breakdown (non-RT)

In [ ]:
for label, df in [('Celeb→Pol', celeb_interactions), ('Pol→Celeb', pol_interactions)]:
    print(f'\n{label} — non-RT language distribution:')
    print(df[df.type != 'rt']['lang'].value_counts().head(10))